---
layout: post 
title: Open Coding Society - Course Exploration 
description: Start interactive experience by pressing "Play".
permalink: /home3-gamified-mvp
codemirror: true
hide: true
toc: false
---


<!-- UI_RUNNER: Data Structures for Courses --> 

<script>

// =======================================
// COURSE OBJECTS - Page-level course data
// =======================================
class Course {
  constructor(code, title, subtitle, mission, unlockThreshold, learningJourney = [], zoneTitle = '') {
    this.code = code;
    this.title = title;
    this.subtitle = subtitle;
    this.mission = mission;
    this.unlockThreshold = unlockThreshold;
    this.learningJourney = learningJourney;
    this.zoneTitle = zoneTitle || `${code} Zone - ${title}`;
  }

  isUnlocked(totalClicks) {
    return totalClicks >= this.unlockThreshold;
  }

  getRemainingClicks(totalClicks) {
    return Math.max(0, this.unlockThreshold - totalClicks);
  }
}

// Define courses as page-level objects
window.ocsCourses = {
  CSSE: new Course(
    'CSSE',
    '1. CSSE - Software Engineering',
    'Games, Foundation',
    'Computer Science and Software Engineering 1,2 prepares students for the AP Computer Science pathway. This course focuses on teaching the JavaScript programming language, object-oriented programming, and developing algorithmic thinking skills through game development projects.',
    5,
    ['Game Builder', 'JS Fundamentals (Code Runner)', 'GitHub Project and VSCode', 'Game Objects', 'Sprite Animation Studio', 'WASD Maze Navigator'],
    'CSSE Zone - Software Engineering Foundation'
  ),
  CSP: new Course(
    'CSP',
    '2. CSP - Computer Science Principles',
    'Projects, Principles',
    'AP Computer Science Principles (CSP 1, CSP 2, DS 1) incorporates College Board curriculum through Project-base learning.  Teams, Data, Coding, Computer Systems, and Networks are utilized to build community serving projects and engage client in discussions about solutions.',
    10,
    ['Python Fundamentals', 'Algorithm Design & Analysis', 'Data Structures Exploration', 'Flask Web Applications', 'SQL Databases', 'Team Project Development'],
    'CSP Zone - Principles of Programming'
  ),
  CSA: new Course(
    'CSA',
    '3. CSA - Computer Science A',
    'Code, Architecture',
    'AP Computer Science A (CSA 1, CSA 2, DS 2) is an in-depth course focusing on programming, algorithms, and data structures. Students prepare for AP exam by architecting and building systems, focusing on code quality, becoming an algorithmic problem solver, establishing fluency in JavaScript, Java and OOP.',
    15,
    ['Java OOP Fundamentals', 'Classes and Objects', 'Inheritance & Polymorphism', 'Arrays & ArrayLists', 'Recursion Patterns', 'Spring Framework Projects'],
    'CSA Zone - Computer Science A'
  ),
  CSH: new Course(
    'CSH',
    '4. CSH - Computer Science Honors',
    'Research, Capstone',
    'Computer Science "H" is a 2-trimester, senior-only interdisciplinary honors course serving as the Pathway Capstone. Students complete a research, design, and coding thesis culminating in a presentation to sponsoring professionals.',
    20,
    ['Capstone Project Planning', 'Research & Problem Definition', 'Stakeholder Engagement', 'System Design & Prototyping', 'Technical Documentation', 'Public Presentation'],
    'CSH Zone - Computer Science Honors'
  )
};

// Expose classes globally for use in other cells
window.Course = Course;

</script>



<!-- UI_RUNNER: Helper classes and data structures for gamified progress tracking and course management --> 

<script>

/** ===============================================
 ZONE STATE MANAGER - Manages zone states 
 ==================================================
 ocsZoneProgress:
 {
    "totalClicks": 20,
    "unlocked": {
        "CSSE": true,
        "CSP": true,
        "CSA": true,
        "CSH": true
    },
    "selectedZone": "CSSE"
}
*/

class ZoneStateManager {
  constructor(storageKey = 'ocsZoneProgress') {
    this.storageKey = storageKey;
    this.totalClicks = 0;
    this.unlocked = {};
    this.selectedZone = null;
    this.load();
  }

  load() {
    const saved = localStorage.getItem(this.storageKey);
    if (saved) {
      try {
        const data = JSON.parse(saved);
        this.totalClicks = data.totalClicks || 0;
        this.unlocked = data.unlocked || {};
        this.selectedZone = data.selectedZone || null;
      } catch (e) {
        console.error('Error loading progress:', e);
      }
    }
  }

  save() {
    localStorage.setItem(this.storageKey, JSON.stringify({
      totalClicks: this.totalClicks,
      unlocked: this.unlocked,
      selectedZone: this.selectedZone
    }));
  }

  incrementClick() {
    this.totalClicks++;
    this.save();
  }

  clear() {
    localStorage.removeItem(this.storageKey);
    this.totalClicks = 0;
    this.unlocked = {};
    this.selectedZone = null;
  }

  selectZone(zoneCode) {
    this.selectedZone = zoneCode;
    this.save();
  }

  isUnlocked(zoneCode) {
    return this.unlocked[zoneCode] || false;
  }

  setUnlocked(zoneCode, unlocked) {
    this.unlocked[zoneCode] = unlocked;
  }
}


// Expose classes globally for use in other cells
window.ZoneStateManager = ZoneStateManager;

</script>


<div id="visionary-interface">
    <div id="world-map-container">
        <img src="{{ site.baseurl }}/images/ocs_world_map.avif" id="main-map">
        
        <div class="island-point" style="top: 35%; left: 18%;" onclick="openMandala('CSSE')"></div>
        <div class="island-point" style="top: 32%; left: 52%;" onclick="openMandala('CSP')"></div>
        <div class="island-point" style="top: 58%; left: 48%;" onclick="openMandala('CSA')"></div>
        <div class="island-point" style="top: 65%; left: 82%;" onclick="openMandala('CSH')"></div>
    </div>

    <div id="mandala-overlay" class="hidden-modal" onclick="closeMandala()">
        <div class="mandala-card" onclick="event.stopPropagation()">
            <button class="close-btn" onclick="closeMandala()">×</button>
            <h2 id="mandala-title">Course Title</h2>
            <h4 id="mandala-subtitle" style="color: #00ffcc; margin-top: 5px;"></h4>
            <p id="mandala-desc" style="margin: 15px 0; font-size: 0.95em;"></p>
            <div style="text-align: left; width: 100%;">
                <strong style="color: #00ffcc; font-size: 0.8em; text-transform: uppercase;">Learning Journey:</strong>
                <ul id="mandala-projects" style="margin-top: 10px; font-size: 0.85em; color: #ccc;"></ul>
            </div>
        </div>
    </div>
</div>

<style>
#world-map-container {
    position: relative;
    width: 100%;
    background: #040d14;
    border: 2px solid #00ffcc55;
    border-radius: 15px;
    overflow: hidden;
    box-shadow: 0 0 20px rgba(0, 255, 204, 0.1);
}

#main-map { width: 100%; display: block; opacity: 0.85; }

.island-point {
    position: absolute;
    width: 16px; height: 16px;
    background: #00ffcc;
    border-radius: 50%;
    cursor: pointer;
    box-shadow: 0 0 15px #00ffcc;
    z-index: 10;
}

/* Sonar Pulse Animation */
.island-point::after {
    content: '';
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: inherit;
    border-radius: 50%;
    animation: sonar 2s infinite;
}

@keyframes sonar {
    0% { transform: scale(1); opacity: 1; }
    100% { transform: scale(4); opacity: 0; }
}

#mandala-overlay {
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0, 0, 0, 0.9);
    display: none; /* Hidden by default */
    justify-content: center;
    align-items: center;
    z-index: 9999;
}

.mandala-card {
    background: #111;
    border: 2px solid #00ffcc;
    border-radius: 50%; /* The Circular Mandala look */
    width: 500px; height: 500px;
    padding: 60px;
    text-align: center;
    color: white;
    display: flex;
    flex-direction: column;
    justify-content: center;
    align-items: center;
    box-shadow: 0 0 40px rgba(0, 255, 204, 0.3);
}

.close-btn {
    position: absolute;
    top: 40px; right: 80px;
    background: none; border: none;
    color: #00ffcc; font-size: 32px;
    cursor: pointer;
}
</style>

<script>
// Initialize the Memory system just in case
if (!window.ocsZoneJourney) {
    window.ocsZoneJourney = new window.ZoneStateManager();
}

window.openMandala = function(courseCode) {
    const course = window.ocsCourses[courseCode];
    if (!course) return;

    // Set text data
    document.getElementById('mandala-title').textContent = course.title;
    document.getElementById('mandala-subtitle').textContent = course.subtitle;
    document.getElementById('mandala-desc').textContent = course.mission;
    
    // Clear and rebuild project list
    const list = document.getElementById('mandala-projects');
    list.innerHTML = '';
    course.learningJourney.forEach(item => {
        const li = document.createElement('li');
        li.textContent = item;
        list.appendChild(li);
    });

    // Show the circular mandala
    document.getElementById('mandala-overlay').style.display = 'flex';
};

window.closeMandala = function() {
    document.getElementById('mandala-overlay').style.display = 'none';
};
</script>